# 03 · The black box, the methods, and the three audits

**The black box is sacred.** It exposes only `predict` / `predict_proba`; no
method here reads a tree, a gradient or a feature importance. Every method sees the
same opaque function and the same feature taxonomy.

Four methods plus two references are compared:

- **CAC** (ours) — jointly optimises causal validity, cost and feasibility.
- **DiCE**, **Wachter**, **Alibi** — established counterfactual methods.
- **Cost oracle** — cheapest flipping plan, ignoring causality (a reference).
- **Random** — a feasible random walk (a floor).

In [ ]:
import sys, pathlib
_here = pathlib.Path.cwd()
ROOT = _here if (_here / "src").exists() else _here.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

RESULTS = ROOT / "results"

def show_image(path):
    """Display a saved figure inline in a notebook; no-op-print in a script."""
    path = pathlib.Path(path)
    try:
        from IPython.display import Image, display  # noqa: PLC0415
        display(Image(filename=str(path)))
    except Exception:
        print("figure saved at", path)

def read_result(name):
    """Read a results CSV, or say which experiment writes it."""
    p = RESULTS / f"{name}.csv"
    if not p.exists():
        raise FileNotFoundError(f"{p} missing - run the experiment that writes it")
    return pd.read_csv(p)

In [ ]:
# How good is the box we are explaining? (trained in experiment 01)
try:
    display(read_result("blackbox_performance"))
except FileNotFoundError as e:
    print(e, "- run experiments/01_train_blackbox.py")

In [ ]:
from src.methods import available_methods
print("registered methods:", available_methods())

## One plan, read the way its recipient would

A counterfactual is not a row of numbers — it is advice. Below is a single CAC plan
for one Adult applicant: what it asks the person to change, and (because CAC is
causal) what the SCM says will follow from it.

In [ ]:
from src.utils.io import load_joblib

res = load_joblib(RESULTS / "counterfactuals_adult.joblib")
record = sorted(res["cac"])[0]
cf = res["cac"][record][0]
print("method:", cf.method, "| record:", record)
print("flips the box:", cf.valid, "| feasible:", cf.feasible,
      "| effort:", round(float(cf.cost), 3))
print("asked to change:", list(cf.changed_features))
cf.frame()  # before / after / changed, per feature

## The three audits, and the gap they open

Every method flips the box (`validity ≈ 1.00`). What separates them is **usable** —
the share of applicants with one plan that flips the box *and* survives the SCM
*and* is something a person could actually do. `usable` is the conjunction of the
three independent audits, and it is where the methods diverge.

In [ ]:
headline = read_result("headline")
cols = [c for c in ["dataset", "method", "validity", "causal_validity",
                    "actionability", "usable", "cost_mean"] if c in headline.columns]
headline[cols].sort_values(["dataset", "usable"], ascending=[True, False])